In [2]:
# =====================
# 기본 설정
# =====================
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Any
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm
from torchvision import transforms

Image.MAX_IMAGE_PIXELS = None
device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
MAX_NEW_TOKENS = 8
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# =====================
# 데이터 로드 (🔥 전체 사용)
# =====================
train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")
train_df = train_df.reset_index(drop=True)

# =====================
# 양자화
# =====================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# =====================
# 프로세서
# =====================
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=384*384,
    max_pixels=768*768,
    trust_remote_code=True,
)

# =====================
# 모델
# =====================
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# =====================
# LoRA
# =====================
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# =====================
# Prompt
# =====================
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Look carefully at the image and choose the correct answer. "
    "Respond with only one letter: a, b, c, or d."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

# =====================
# Dataset (🔥 augmentation 추가)
# =====================
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

        self.transform = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
        ])

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        if self.train:
            img = self.transform(img)

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]

        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}

# =====================
# Collator (🔥 label masking 핵심)
# =====================
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []

        for sample in batch:
            text = self.processor.apply_chat_template(
                sample["messages"],
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)
            images.append(sample["image"])

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        if self.train:
            labels = enc["input_ids"].clone()

            for i, text in enumerate(texts):
                # assistant 이전 부분 mask
                split_token = "assistant"
                idx = text.rfind(split_token)
                if idx != -1:
                    tokenized = self.processor.tokenizer(
                        text[:idx],
                        return_tensors="pt"
                    )
                    cutoff = tokenized["input_ids"].shape[1]
                    labels[i, :cutoff] = -100

            enc["labels"] = labels

        return enc

# =====================
# 데이터 split
# =====================
split = int(len(train_df)*0.9)
train_subset = train_df.iloc[:split]
valid_subset = train_df.iloc[split:]

train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True,
                          collate_fn=DataCollator(processor, True))
valid_loader = DataLoader(valid_ds, batch_size=2, shuffle=False,
                          collate_fn=DataCollator(processor, True))

# =====================
# 학습 설정
# =====================
model = model.to(device)
GRAD_ACCUM = 8
num_epochs = 3

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

num_training_steps = num_epochs * math.ceil(len(train_loader)/GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    int(num_training_steps*0.03),
    num_training_steps
)

scaler = torch.cuda.amp.GradScaler(enabled=True)

# Early stopping
best_loss = float("inf")
patience = 1
counter = 0

# =====================
# 학습 루프
# =====================
for epoch in range(num_epochs):
    model.train()
    running = 0.0

    for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}")):
        batch = {k:v.to(device) for k,v in batch.items()}

        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            loss = model(**batch).loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if (step+1) % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

    # =====================
    # Validation
    # =====================
    model.eval()
    val_loss = 0.0
    val_steps = 0

    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        for vb in valid_loader:
            vb = {k:v.to(device) for k,v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1

    val_loss /= val_steps
    print(f"[Epoch {epoch+1}] val_loss: {val_loss:.4f}")

    # Early stopping
    if val_loss < best_loss:
        best_loss = val_loss
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered")
            break

# =====================
# 저장
# =====================
SAVE_DIR = "/content/qwen2_5_vl_7b_lora"
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

print("Saved:", SAVE_DIR)

c:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SSAFY\.cache\huggingface\hub\models--Qwen--Qwen2.5-VL-7B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__

RuntimeError: We encountered some issues during automatic conversion of the weights. For details look at the `CONVERSION` entries of the above report!

In [ ]:
# 데이터 파서 : 모델의 응답에서 선지를 추출
def extract_choice(text: str) -> str:
    text = text.strip().lower()

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last

    tokens = last.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]:
            return tok
    return "a"

# 추론을 위해 모든 레이어 활성화
model.eval()
preds = []

# 추론 루프
for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    img = Image.open(row["path"]).convert("RGB")
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[
            {"type":"image","image":img},
            {"type":"text","text":user_text}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=2, do_sample=False,
                                 eos_token_id=processor.tokenizer.eos_token_id)
    output_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    # print("output_text:", output_text)
    # print("extract_choice:", extract_choice(output_text))
    preds.append(extract_choice(output_text))

# 제출 파일 생성
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission.csv", index=False)
print("Saved /content/submission.csv")